In [1]:
# import require library for preprocess
import mne
import numpy as np
from mne.channels import make_standard_montage
import matplotlib.pyplot as plt
from mne.datasets import eegbci
import scipy
import pickle
import seaborn as sns
from scipy.signal import filtfilt
import pyxdf

# import require library for classification
from sklearn.svm import SVC # SVM library
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis # LDA library
from sklearn.neighbors import KNeighborsClassifier # KNN library

from sklearn.metrics import classification_report,confusion_matrix # Result representation

In [25]:
import pyxdf
import mne
import numpy as np
streams, header = pyxdf.load_xdf("C:\\Users\\pipo_\\OneDrive\\Desktop\\neuromedia\\biosemiLSL.xdf") #Example Data from Lab Recoder

raw_data = streams[0]["time_series"].T #From Steam variable this query is EEG data
with open("datasets/biosemi_chans.pkl", "rb") as file:
    ch_names = pickle.load(file)


info = mne.create_info(
    ch_names= ch_names,   
    ch_types= ['eeg']*len(ch_names),
    sfreq= 512 #OpenBCI Frequency acquistion
)
raw_OpenBCI = mne.io.RawArray(raw_data[0:73], info, verbose=False)    


In [26]:
raw = raw_OpenBCI.copy().filter(l_freq=0.16, h_freq=100, method= 'iir', iir_params= dict(order=8, ftype='butter'))

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.16 - 1e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 32 (effective, after forward-backward)
- Cutoffs at 0.16, 100.00 Hz: -6.02, -6.02 dB



In [27]:
raw.get_data()[20:40,:] #Get data from raw object

array([[  0.10335324,   0.249886  ,   0.02923069, ...,  -0.20308525,
          0.79114109,   0.58310391],
       [ -0.44947762,  -3.33079061,  -2.85469966, ...,   4.60707776,
          3.65845314,   1.61259944],
       [ -1.36096093,  -4.64814436,   0.82723154, ...,   2.12811311,
          2.6981615 ,   1.78292886],
       ...,
       [  0.24240475,  -0.38713866,  -3.6828515 , ...,  -2.78805426,
         -3.50646115,  -3.82154374],
       [  1.60277325,   4.97175766,   4.92339627, ..., -10.34706106,
         -7.54892787,  -4.34727199],
       [  1.4657803 ,   3.94303118,   2.85833801, ...,  -2.61989398,
         -3.16518234,  -4.40613668]])

In [18]:
raw_data = streams[0]["time_series"].T
raw_data[0:64]

array([[-7.7624900e+06, -7.7624900e+06, -7.7624900e+06, ...,
        -7.7624900e+06, -7.7624900e+06, -7.7624900e+06],
       [-4.1724028e+03, -4.1726577e+03, -4.1804180e+03, ...,
        -4.0817095e+03, -4.0834272e+03, -4.0852920e+03],
       [-1.5657633e+04, -1.5658581e+04, -1.5667202e+04, ...,
        -1.5592025e+04, -1.5581305e+04, -1.5595889e+04],
       ...,
       [ 2.0463840e+03,  1.9900858e+03,  1.9645870e+03, ...,
         2.1298845e+03,  2.1346243e+03,  2.1245027e+03],
       [-3.0301465e+03, -3.0237007e+03, -3.0323979e+03, ...,
        -2.9563950e+03, -2.9590454e+03, -2.9677766e+03],
       [-5.5254233e+03, -5.5224795e+03, -5.5278267e+03, ...,
        -5.5130840e+03, -5.5115166e+03, -5.5157861e+03]], dtype=float32)

In [20]:
from scipy import signal

def butter_bandpass(lowcut,highcut,fs,order):
    nyq = 0.5*fs
    low = lowcut/nyq
    high = highcut/nyq
    b,a = signal.butter(order,[low,high],'bandpass')
    return b,a

def butter_bandpass_filter(data,lowcut = 6, highcut = 30, order = 4, axis = 1):
    b,a = butter_bandpass(lowcut,highcut,512,order)
    y = signal.filtfilt(b,a,data,axis=axis)
    return y

filtered_data = butter_bandpass_filter(raw_data[0:64].reshape(1,64, raw_data.shape[1]), lowcut = 0.16, highcut= 100, axis = 2)

In [21]:
filtered_data

array([[[ 6.18258391e-01,  6.39157570e-01,  6.60051052e-01, ...,
         -1.82949229e-03, -1.75504520e-03, -1.68065190e-03],
        [-1.89766398e+01, -2.13929915e+01, -2.62549538e+01, ...,
         -1.65628719e+01, -1.81025541e+01, -1.93629527e+01],
        [-2.20557495e+01, -2.56947118e+01, -3.02102015e+01, ...,
         -9.01405058e+00, -1.30606479e+01, -1.66724965e+01],
        ...,
        [ 1.95460245e+01, -2.86717667e+01, -4.59703530e+01, ...,
         -2.00303481e+01, -2.12905220e+01, -2.57102825e+01],
        [ 4.77711934e+00,  6.45387491e+00,  3.88689200e+00, ...,
         -7.62425104e+00, -9.39732283e+00, -1.70014857e+01],
        [-1.62567477e+00, -1.38907939e+00, -3.24574704e+00, ...,
         -4.60253670e+00, -4.61320629e+00, -7.46909902e+00]]])

In [6]:
with open("datasets/biosemi_chans.pkl", "rb") as file:
    ch_names = pickle.load(file)
ch_names

['Fp1',
 'AF7',
 'AF3',
 'F1',
 'F3',
 'F5',
 'F7',
 'FT7',
 'FC5',
 'FC3',
 'FC1',
 'C1',
 'C3',
 'C5',
 'T7',
 'TP7',
 'CP5',
 'CP3',
 'CP1',
 'P1',
 'P3',
 'P5',
 'P7',
 'P9',
 'PO7',
 'PO3',
 'O1',
 'Iz',
 'Oz',
 'POz',
 'Pz',
 'CPz',
 'Fpz',
 'Fp2',
 'AF8',
 'AF4',
 'AFz',
 'Fz',
 'F2',
 'F4',
 'F6',
 'F8',
 'FT8',
 'FC6',
 'FC4',
 'FC2',
 'FCz',
 'Cz',
 'C2',
 'C4',
 'C6',
 'T8',
 'TP8',
 'CP6',
 'CP4',
 'CP2',
 'P2',
 'P4',
 'P6',
 'P8',
 'P10',
 'PO8',
 'PO4',
 'O2',
 'EXG1',
 'EXG2',
 'EXG3',
 'EXG4',
 'EXG5',
 'EXG6',
 'EXG7',
 'EXG8',
 'STATUS']